<a href="https://colab.research.google.com/github/mehluli92/time_series_forecasting/blob/customer_complaints_forecasting/Holt_Winters.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# SetUp

In [ ]:
# Mount the drive
#from google.colab import drive
#drive.mount('/content/drive')

In [ ]:
%cd /content/drive/MyDrive/Colab Notebooks/Python - Time Series Forecasting/Time Series Analysis/Exponential Smoothing and Holt Winters

In [ ]:
# Import libraries
import pandas as pd
import matplotlib.pyplot as plt
from statsmodels.graphics.tsaplots import month_plot, quarter_plot, plot_acf, plot_pacf
from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.tsa.holtwinters import ExponentialSmoothing, SimpleExpSmoothing
from sklearn.metrics import root_mean_squared_error, mean_absolute_error, mean_absolute_percentage_error

In [ ]:
# Set index when importing the data
df = pd.read_csv('weekly_customer_complaints.csv', index_col="week", parse_dates=True)
df.head()

In [ ]:
# Change the name of the time series variable to y
df = df.rename( columns= {"complaints": "y"})
df.head()

In [ ]:
df.info()

In [ ]:
# Remove comma from df.y and transform into an integer
df["y"] = df["y"].str.replace(",", "").astype(int)
df.head()

# Data Visualization

In [ ]:
# Time Series plot
df["y"].plot()
plt.show()

# Seasonality

In [ ]:
# Quater Plot the month_plot for df
month_plot(df['y'].resample('ME').mean())
plt.show()

In [ ]:
# Plot the quater_plot
quarter_plot(df['y'].resample('QE').mean())
plt.show()

In [ ]:
# Seasonal decomposition for df['Adj.Close']
decomposition = seasonal_decompose(df['y'],
                                   model= 'multiplicative',
                                   period = 52
                                   )
fig = decomposition.plot()
fig.set_size_inches(18, 10)
plt.show()

# Auto-correlation

In [ ]:
# Plot the ACF of the Bitcoin Adj Close
fig, ax = plt.subplots(figsize = (12, 6))
plot_acf(df['y'], lags= 100, ax = ax)
plt.show()

# (Partial) Auto-Correlation

In [ ]:
# PACF for bitcoin adj close
fig, ax = plt.subplots(figsize= (12, 6))
plot_pacf(df['y'], lags= 100, ax = ax)
plt.show()

# Time Series Frequency

In [ ]:
# Print the frequency of the time series
df.index

In [ ]:
# Change the frequency to W-Mon
df = df.asfreq('W-Mon')
df.index

# Training and Test

# GOAL: Predict the next 13 weeks

In [ ]:
# Split the data into training and test
periods = 13
train = df[:-periods].y
test = df[-periods:].y

# Other way to split the data
# train, test = df.iloc[:-periods, 0], df.iloc[-periods:, 0]

In [ ]:
train.head()

# Simple Exponential smoothing

Equation:

current = current + alpha * observed

In [ ]:
# Apply SES to the train dataset
ses_model = SimpleExpSmoothing(train).fit()
print(ses_model.summary())

In [ ]:
# Predictions
ses_pred = ses_model.forecast(periods)
ses_pred

In [ ]:
# Set the size of the plot to 10 by 4
plt.figure(figsize= (10,4))

# Plot the train, test and forecast data
plt.plot(train.loc['2022'], label = "Train")
plt.plot(test, label = "Test")
plt.plot(ses_pred, label = "Forecast")

# Add a Title and legend to the plot
plt.title("Simple Exponential Smoothing")
plt.legend()
plt.show()

# Double Exponential Smoothing

It uses simple smoothing and the trand of the data.

Smoothed_level = alpha * Recent Actual + (1-alpha) * (Previous_level - Previous_trend)

Smoothed_trend = Beta * (Smoothed_level - Previous_level) + (1 - Beta) * Previous_trend

In [ ]:
# Build double exponential smoothing model
model_double = ExponentialSmoothing(
    endog = train,
    trend = "add",
    seasonal = None).fit()

print(model_double.summary())

In [ ]:
# Predict using double exponential smoothing model
double_pred = model_double.forecast(periods)
double_pred

In [ ]:
# Plot the train, test and forecast
plt.figure(figsize= (10,4))

# Plot the train, test and forecast data
plt.plot(train.loc['2022'], label = 'Train')
plt.plot(test, label = 'Test')
plt.plot(double_pred, label = "Forecast")

# Add title and legend
plt.title('Double Exponential Smoothing')
plt.legend()
plt.show()

# Holt-Winters / Tripple Exponential Smoothing

- For data with both trends and seasonality
- Data is split into three level, trend and seasonality

In [ ]:
# Build the Holt-Winters model
model_holt = ExponentialSmoothing(
    endog = train,
    trend = 'add',
    seasonal = 'mul',
    seasonal_periods = 52).fit()

print(model_holt.summary())

In [ ]:
# Predict with Holt-Winters model
holt_pred = model_holt.forecast(periods)
holt_pred

In [ ]:
# Plot the train, test and the forecast
plt.figure(figsize=(10,6))

# Plot the train, test and forecast
plt.plot(train.loc['2022'], label = 'Train')
plt.plot(test, label = 'Test')
plt.plot(holt_pred, label = 'Forecast')

# Add a title and legend to the plot
plt.title('Holt-Winters')
plt.legend()
plt.show()

# Error calculation for RMSE, MAE and MAPE

In [ ]:
# Error calculation for RMSE, MAE and MAPE
rmse = root_mean_squared_error(test, holt_pred)
mae = mean_absolute_error(test, holt_pred)
mape = mean_absolute_percentage_error(test, holt_pred)

print(f'RMSE: {rmse: .0f}')
print(f'MAE: {mae: .0f}')
print(f'MAPE: {100 * mape:.1f} %')


In [ ]:
# Function that assesses the model and visualizes the train, test and forecast.
def model_assessment(train, test, predictions, chart_title = None):
  # Set the size of the plot of 10 by 4
  plt.figure(figsize= (10,4))

  # Plot the train, test and forecast data
  plt.plot(train, label = 'Train')
  plt.plot(test, label = 'Test')
  plt.plot(predictions, label = 'Forecast')
  plt.title(chart_title)
  plt.legend()
  plt.show()

  # Error calculation for RMSE, MAE and MAPE
  # Error calculation for RMSE, MAE and MAPE
  rmse = root_mean_squared_error(test, predictions)
  mae = mean_absolute_error(test, predictions)
  mape = mean_absolute_percentage_error(test, predictions)

  print(f'RMSE: {rmse: .0f}')
  print(f'MAE: {mae: .0f}')
  print(f'MAPE: {100 * mape:.1f} %')

# Apply the function to the problem
model_assessment(train.loc['2022'], test, holt_pred, "Holt-Winters")

# Predict the future

In [ ]:
# Build a Holt-Winters model with the complete data
# Build the Holt-Winters model
model_holt_complete = ExponentialSmoothing(
    endog = df.y,
    trend = 'add',
    seasonal = 'mul',
    seasonal_periods = 52).fit()

In [ ]:
# Predict with the model
forecast = model_holt_complete.forecast(13)
forecast[:5]

In [ ]:
def plot_future(y, forecast, chart_title= None):
  # Set chart size
  plt.figure(figsize= (12, 5))

  # Plot the train, test and forecast
  plt.plot(y, label = ('Train'))
  plt.plot(forecast, label = 'Forecast')

  # Add a title and legend to the plot
  plt.title(chart_title)
  plt.legend()
  plt.show()

plot_future(df['y'].loc["2022"], forecast, "Holt-Winters")

# Daily Data